<a href="https://colab.research.google.com/github/rxphaelbihag/Linear-Programming/blob/main/Store_Location_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Store Location Optimization

DS115: End-of-Sem Project

## Description
In this project, you are an entrepreneur with a budget enough to open 2 convenience stores in downtown Davao. The idea is to find the best locations of only two convenience stores that would allow more access to your target clients. You are provided 7 locations to choose from and you are given the locations of your 5 target clients.

## Specifications
1. The coordinates of the 7 possible store locations are stored in a .CSV file.
2. The coordinates of the 5 clients are stored in a .CSV file.
3. Randomly pick two pairs of possible store locations.
4. Using p-center as the math model for this problem, determine which of the
two sets should you choose as locations for your stores
5. Map the clients and the winning set stores

## Calculations
### Libraries and CSV files
We use `pandas` as our main tool for data manipulation. The `cdist` class from `scipy` allows us to calculate eucledian distances. We use `random` to generate random numbers. We use `folium` to generate the map of the final answer.

In [50]:
import pandas as pd
from scipy.spatial.distance import cdist
import random
import folium

### Import the CSV Files

In [51]:
store_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/bihag_storelocs.csv"
client_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/client_locs.csv"

stores = pd.read_csv(store_locations_csv, index_col='store')
clients = pd.read_csv(client_locations_csv, index_col='client')

In [52]:
stores

,latitude,longitude
store,,
1,7.0901382366,125.6067659888
2,7.0887236013,125.6308328384
3,7.0816559392,125.6128427311
4,7.0818693404,125.6212218350
5,7.0792195616,125.6250239650
6,7.0720149602,125.6028713497
7,7.0656564747,125.6106411320


In [53]:
clients

,latitude,longitude
client,,
1,7.0867331199,125.6218158294
2,7.0826867149,125.6157029373
3,7.0772301121,125.6047689925
4,7.0851514811,125.6070864206
5,7.0732906709,125.6118787665


### Construct Distance Matrix

We use a distance matrix to calculate the distance of each client to each store. This is so that it will be easier for calculations later.

In [54]:
# Extract coordinate columns as NumPy arrays
client_coords = clients[['latitude', 'longitude']].values
store_coords = stores[['latitude', 'longitude']].values

# Compute the pairwise distance matrix
distance_array = cdist(client_coords, store_coords, metric='euclidean')

# Convert the array back into a labeled Pandas DataFrame
distance_matrix = pd.DataFrame(
    distance_array,
    index=clients.index,  # Rows represent clients
    columns=stores.index  # Columns represent stores
)

distance_matrix

store,1,2,3,4,5,6,7
client,,,,,,,
1,0.0154302470,0.0092340927,0.0103099106,0.0048999163,0.0081698036,0.0239899466,0.0238557924
2,0.0116359024,0.0162898098,0.0030402760,0.0055790978,0.0099449841,0.0166893975,0.0177665684
3,0.0130616872,0.0284855113,0.0092072363,0.0170943986,0.0203524401,0.0055496719,0.0129781009
4,0.0049970398,0.0240135878,0.0067345322,0.0145114571,0.0188929396,0.0137961953,0.0198164388
5,0.0176062763,0.0244424259,0.0084206260,0.0126841042,0.0144204018,0.0090973070,0.0077338665


### Random Two Sets
We randomly pick two sets or two pairs of store locations. The store locations are indexed from 1 to 7 (Store 1 to Store 7). So, we use `random`'s `sample()` function to sample four random numbers and group them to two. This is to ensure that the two sets do not have the same stores.

In [72]:
numbers = random.sample(range(1, 8), 4)

# The sets in index form
set1 = [numbers[0], numbers[1]]
set2 = [numbers[2], numbers[3]]

# The sets in coordinates form
set1_coords = [store_coords[set1[0]-1], store_coords[set1[1]-1]]
set2_coords = [store_coords[set2[0]-1], store_coords[set2[1]-1]]

print("Set 1")
print(f"{'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*40)
print(f"Store {set1[0]:<1} | {set1_coords[0][0]:<10.10f} | {set1_coords[0][1]:<10.10f}")
print(f"Store {set1[1]:<1} | {set1_coords[1][0]:<10.10f} | {set1_coords[1][1]:<10.10f}")

print("\n\nSet 2")
print(f"{'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*40)
print(f"Store {set2[0]:<1} | {set2_coords[0][0]:<10.10f} | {set2_coords[0][1]:<10.10f}")
print(f"Store {set2[1]:<1} | {set2_coords[1][0]:<10.10f} | {set2_coords[1][1]:<10.10f}")

Set 1
Store # | Latitude     | Longitude      
----------------------------------------
Store 3 | 7.0816559392 | 125.6128427311
Store 2 | 7.0887236013 | 125.6308328384


Set 2
Store # | Latitude     | Longitude      
----------------------------------------
Store 1 | 7.0901382366 | 125.6067659888
Store 7 | 7.0656564747 | 125.6106411320


### Solution Evaluation
Next, we evaluate the distances of each of the clients to each of the stores in our two random sets. This will be very easy since we have already calculated all of the pairwise distances and stored in the `distance_matrix`. Now, we just have to evaluate each client's distance to the two stores in a set to see which store is closest to it and by how much.

In [56]:
# Evaluating first set
"""
Stores the distances. Format: "Client 1": [<Distance>, <Closest Store>]
- Distance is in degrees
- Closest Store is either 0 if closest to the first store in the set, 1 if otherwise
"""
set1_dists = {}
set2_dists = {}

for client in range(1, 5+1):
    s1store1 = set1[0]    # index of store1 in set1
    s1store2 = set1[1]    # index of store2 in set1
    s2store1 = set2[0]    # index of store1 in set2
    s2store2 = set2[1]    # index of store2 in set2

    s1distto_s1 = distance_matrix[s1store1][client] # dist of client to store1,set1
    s1distto_s2 = distance_matrix[s1store2][client] # dist of client to store2,set1
    s2distto_s1 = distance_matrix[s2store1][client] # dist of client to store1,set2
    s2distto_s2 = distance_matrix[s2store2][client] # dist of client to store2,set2

    # find the minimum among the two distances
    if s1distto_s1 > s1distto_s2:
        set1_dists[f"client{client}"] = (s1distto_s2, 1)
    else:
        set1_dists[f"client{client}"] = (s1distto_s1, 0)

        # find the minimum among the two distances
    if s2distto_s1 > s2distto_s2:
        set2_dists[f"client{client}"] = (s2distto_s2, 1)
    else:
        set2_dists[f"client{client}"] = (s2distto_s1, 0)

In [57]:
# Print Set 1 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in set1_dists.items():
    store_name = set1[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.00923409   | 2         
client2    | 0.01628981   | 2         
client3    | 0.00554967   | 6         
client4    | 0.01379620   | 6         
client5    | 0.00909731   | 6         


In [58]:
# Print Set 2 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in set2_dists.items():
    store_name = set2[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.00489992   | 4         
client2    | 0.00557910   | 4         
client3    | 0.01306169   | 1         
client4    | 0.00499704   | 1         
client5    | 0.01268410   | 4         


In [59]:
# Evaluate which set is the best
if max(set1_dists.values())[0] > max(set2_dists.values())[0]:
    best = set2
    print("The best set is Set 2:", set2, "The max distance (deg) is", max(set2_dists.values())[0])
else:
    best = set1
    print("The best set is Set 1:", set1, "The max distance (deg) is", max(set1_dists.values())[0])

The best set is Set 2: [4, 1] The max distance (deg) is 0.013061687201905299


### Optimal Solution

Now that we have identified the best set. We plot them on a map to visualize it.

In [60]:
# Initialize map center (Somewhere in downtown Davao City)
mymap = folium.Map(location=[7.080506387226319, 125.6120560701777],
                   zoom_start=15)

# dictionary of client coordinates
clients = {}
for _ in range(len(client_coords)):
    clients[f"Client {_+1}"] = [client_coords[_][0], client_coords[_][1]]

# dictionary of store coordinates in the best set
best_set = {}
for _ in range(2):
    best_set[f"Store {best[_]}"] = [store_coords[best[_]-1][0],
                                    store_coords[best[_]-1][1]]

# to determine the max radius in the best set
best_dists = set2_dists if best == set2 else set1_dists
max_radius_degrees = max(best_dists.values())[0]

# Draws the clients on the map
for client, coords in clients.items():
    folium.Marker(
        location=coords,
        popup=client,
        icon=folium.Icon(color="blue", icon="person", prefix='fa')
    ).add_to(mymap)

# Draws the stores on the map
for store, coords in best_set.items():
    folium.Marker(
        location=coords,
        popup=store,
        icon=folium.Icon(color="red", icon="store", prefix='fa')
    ).add_to(mymap)

    # Draws the circles around the stores of the best set
    folium.Circle(
        location=coords,
        radius=max_radius_degrees * 111320, # Radius from deg to meters
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.2,
        popup="Coverage Area"
    ).add_to(mymap)

mymap

# Genetic Algorithm Application
Below is the extension of the optimization problem above by applying Genetic Algorithm to the solutions.

## Specifications
1. Apply the crossover operator on the 2 sets of locations. By using a single point crossover, randomly select a crossover point and generate two offspring by swapping the 1st part of parent1 with 1st part of parent2.   
2. Based on the two new offsprings (after crossover operator), apply single point mutation on each of the new offspring. For offspring1, randomly select a single mutation point and mutate the gene in that location by adding 0.01 (or 1 x 10-2) in that location. For offspring2, select another random mutation point (preferably different from the point chosen in offspring1 and mutate that gene as well by adding 0.001 or (1 x 10-3) in that location.
3. Calculate the fitness value of each of the new offspring.
4. Compare the performance of the two new offspring with the parents.

## Calculations

### Crossover Operator
In this case, we apply the crossover operator by simply swapping the tails of the two sets.

For example:
$$\text{Parent 1}=(store_{11}, store_{12})$$
$$\text{Parent 2}=(store_{21}, store_{22})$$
$$\text{Offspring 1}=(store_{11}, store_{22})$$
$$\text{Offspring 2}=(store_{21}, store_{12})$$

In [112]:
# Print the parents
print("-- Parents --")
print("Parent 1 (Set 1):", set1)
print("Parent 2 (Set 2):", set2)

# Execute crossover
print("\n## Crossover ##\n")
offspring1, offspring2 = [set1[0], set2[1]], [set2[0], set1[1]]
offspring1_coords = [store_coords[offspring1[0]-1], store_coords[offspring1[1]-1]]
offspring2_coords = [store_coords[offspring2[0]-1], store_coords[offspring2[1]-1]]

# Print the offspring
print("-- Offspring --")
print("Offsping 1:", offspring1)
print("Offsping 2:", offspring2)

-- Parents --
Parent 1 (Set 1): [3, 2]
Parent 2 (Set 2): [1, 7]

## Crossover ##

-- Offspring --
Offsping 1: [3, 7]
Offsping 2: [1, 2]



### Mutation Operator
On the two offspring after crossover, we apply single point mutation on each of the new offspring. For offspring1, we randomly select a single mutation point and mutate the gene in that location by adding 0.01 $(\text{or }1\times 10^{-2})$ in that location. For offspring2, select another random mutation point (preferably different from the point chosen in offspring1 and mutate that gene as well by adding 0.001 or($(1 \times 10^{-3})$ in that location.

For example:
Let random integer 1 or 0 be $z=1$. Then,
$$\text{Offspring 1} = (store_{11}, store_{12}+0.01)$$
$$\text{Offspring 2} = (store_{21}+0.001, store_{22})$$

In [113]:
print("Before Mutation")
print(f"{'Offspring #':<1} | {'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*60)
print(f"{'Offspring 1':<1} | Store {offspring1[0]:<1} | {offspring1_coords[0][0]:<10.10f} | {offspring1_coords[0][1]:<10.10f}")
print(f"{'Offspring 1':<1} | Store {offspring1[1]:<1} | {offspring1_coords[1][0]:<10.10f} | {offspring1_coords[1][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[0]:<1} | {offspring2_coords[0][0]:<10.10f} | {offspring2_coords[0][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[1]:<1} | {offspring2_coords[1][0]:<10.10f} | {offspring2_coords[1][1]:<10.10f}")

mutation_point = random.choice([0, 1])

offspring1_coords[mutation_point] = offspring1_coords[mutation_point]+0.01
offspring2_coords[1 if mutation_point==0 else 0] = offspring2_coords[1 if mutation_point==0 else 0]+0.001


print("\n## Mutation ##")
print("Mutation at Gene index", mutation_point)

print("\nAfter Mutation")
print(f"{'Offspring #':<1} | {'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*60)
print(f"{'Offspring 1':<1} | Store {offspring1[0]:<1} | {offspring1_coords[0][0]:<10.10f} | {offspring1_coords[0][1]:<10.10f}")
print(f"{'Offspring 1':<1} | Store {offspring1[1]:<1} | {offspring1_coords[1][0]:<10.10f} | {offspring1_coords[1][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[0]:<1} | {offspring2_coords[0][0]:<10.10f} | {offspring2_coords[0][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[1]:<1} | {offspring2_coords[1][0]:<10.10f} | {offspring2_coords[1][1]:<10.10f}")

Before Mutation
Offspring # | Store # | Latitude     | Longitude      
------------------------------------------------------------
Offspring 1 | Store 3 | 7.0816559392 | 125.6128427311
Offspring 1 | Store 7 | 7.0656564747 | 125.6106411320
Offspring 2 | Store 1 | 7.0901382366 | 125.6067659888
Offspring 2 | Store 2 | 7.0887236013 | 125.6308328384

## Mutation ##
Mutation at Gene index 0

After Mutation
Offspring # | Store # | Latitude     | Longitude      
------------------------------------------------------------
Offspring 1 | Store 3 | 7.0916559392 | 125.6228427311
Offspring 1 | Store 7 | 7.0656564747 | 125.6106411320
Offspring 2 | Store 1 | 7.0901382366 | 125.6067659888
Offspring 2 | Store 2 | 7.0897236013 | 125.6318328384


### Offspring Evaluation and Comparison
We will now calculate the maximum distance of each of the offspring to their furthest client. First, we create another tempororary distance matrix for easy comparison.

If elitism is applied, will any of the offspring replace any of the parents.

Support your answer with an explanation why you will replace or why
you will not replace them.